<a href="https://colab.research.google.com/github/PrakharDoneria/AI-Assistant/blob/main/Viber_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup enviroment

In [1]:
# Check GPU
!nvidia-smi

# Install llama-cpp and Hugging Face Hub
!pip install llama-cpp-python huggingface_hub --upgrade

/bin/bash: line 1: nvidia-smi: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 MB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.8-cp311-cp311-linux_x86_64.whl size=5959614 sha256=c1bf830bed8af6c637585d35615c44129939f189a3ce10a00b8aebc9eabedbc9
  Stored in directory: /root/.cache/pip/wheels/c0/03/66/eb3810eafd55d921b2be32896d1f44313996982360663aa80b
Successfully built llama-cpp-python


# Download Model

In [2]:
from google.colab import userdata
hf_token = userdata.get("colabagent")
print("✅ Token retrieved:", hf_token[:5] + "...")

✅ Token retrieved: hf_QP...


In [3]:
from huggingface_hub import hf_hub_download
from google.colab import userdata

# Load your Hugging Face token securely
hf_token = userdata.get("colabagent")

# Download the GGUF model using the correct repository ID
model_file = hf_hub_download(
    repo_id="TheBloke/deepseek-coder-1.3b-instruct-GGUF",
    filename="deepseek-coder-1.3b-instruct.Q4_K_M.gguf",
    token=hf_token
)

print("✅ Model downloaded to:", model_file)

deepseek-coder-1.3b-instruct.Q4_K_M.gguf:   0%|          | 0.00/874M [00:00<?, ?B/s]

✅ Model downloaded to: /root/.cache/huggingface/hub/models--TheBloke--deepseek-coder-1.3b-instruct-GGUF/snapshots/4595af8c3dff738094bd6c86054dfb5a90d5c41e/deepseek-coder-1.3b-instruct.Q4_K_M.gguf


# Load AI Model

In [4]:
from llama_cpp import Llama
import torch

# Detect if CUDA GPU is available using PyTorch (you can also use another method if PyTorch isn't available)
gpu_available = torch.cuda.is_available()

# Adjust GPU usage depending on availability
n_gpu_layers = 35 if gpu_available else 0  # 0 means all layers run on CPU

# Load the model
llm = Llama(
    model_path=model_file,
    n_ctx=2048,
    n_threads=8,
    n_gpu_layers=n_gpu_layers,
    use_mlock=False
)

llama_model_loader: loaded meta data with 22 key-value pairs and 219 tensors from /root/.cache/huggingface/hub/models--TheBloke--deepseek-coder-1.3b-instruct-GGUF/snapshots/4595af8c3dff738094bd6c86054dfb5a90d5c41e/deepseek-coder-1.3b-instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = deepseek-ai_deepseek-coder-1.3b-instruct
llama_model_loader: - kv   2:                       llama.context_length u32              = 16384
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          llama.block_count u32              = 24
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 5504
ll

# Interaction part

In [ ]:
import os
import subprocess
import time
import re

# Function to create a directory based on timestamp for uniqueness
def create_directory():
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    folder_name = f"generated_code_{timestamp}"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    print(f"✅ Directory '{folder_name}' created.")
    return folder_name

# Function to check if a file exists, and if it does, load its content
def load_existing_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            return f.read()
    return ""

# Function to save generated content to a specified file
def save_to_file(file_path, content):
    with open(file_path, "w") as f:
        f.write(content)
    print(f"✅ Code saved to {file_path}")

# Function to lint HTML code (basic validation for now)
def lint_html(code):
    # For simplicity, we'll just check if the code contains a `</html>` tag (basic validation)
    if "</html>" not in code:
        print("❌ Linting error: Missing '</html>' tag.")
        return False
    print("✅ No HTML linting errors found.")
    return True

# Function to extract only HTML code from the AI response
def extract_html_code(ai_response):
    # Enforce HTML code output by wrapping with <html> if needed.
    if "html" not in ai_response.lower():
        print("❌ Expected HTML, but received a non-HTML response. Converting it to HTML.")
        return f"<html><body><pre>{ai_response}</pre></body></html>"
    return ai_response.strip()

# Function to handle code modification, saving, and running (for web files)
def update_web_code_and_run(code, folder_name, file_type="html"):
    # Define the file path and create it in the specified folder
    filename = f"generated_page.{file_type}"
    file_path = os.path.join(folder_name, filename)

    # Save the generated code to the file
    save_to_file(file_path, code)

    # Lint the code (for HTML files)
    if lint_html(code):
        print(f"🚀 File '{filename}' is ready for viewing in the browser!")
    else:
        print("❌ File not ready due to errors.")

# Function to handle the whole code generation, update, and modification process
def continuous_web_chat_and_update_code(instruction):
    # Generate initial code based on the first instruction
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\nPlease provide only the HTML code for the following task: {instruction}"

    # Generate HTML code using Llama model
    response = llm(prompt, max_tokens=512, stop=["###"])
    generated_code = response['choices'][0]['text']

    # Extract only HTML code from the AI response
    cleaned_code = extract_html_code(generated_code)

    # Show generated code (for reference)
    print("\n🧠 Generated HTML Code:\n")
    print(cleaned_code)

    # Automatically create folder based on timestamp
    folder_name = create_directory()

    # Automatically save the initial HTML code
    update_web_code_and_run(cleaned_code, folder_name)

    # Continuous chat loop for user input and code modification
    while True:
        user_input = input("\n💬 What would you like to modify or add? (Type 'exit' to quit, 'debug' to check errors): ")

        if user_input.lower() == 'exit':
            print("👋 Exiting the chat.")
            break

        if user_input.lower() == 'debug':
            print("🔍 Running a debug check on the current HTML code...")
            lint_html(cleaned_code)
            continue

        # Format prompt for AI response
        prompt = f"### Instruction:\n{user_input}\n\n### Response:\nPlease provide only the HTML code for this task."

        # Generate new HTML code or modification based on user input
        response = llm(prompt, max_tokens=512, stop=["###"])
        modification_code = response['choices'][0]['text']

        # Extract only HTML code from the response
        modified_code = extract_html_code(modification_code)

        # Combine the existing code with the new modification (assumes append-like behavior)
        cleaned_code += "\n" + modified_code

        # Show updated HTML code
        print("\n🧠 Updated HTML Code:\n")
        print(cleaned_code)

        # Update and run the modified HTML code
        update_web_code_and_run(cleaned_code, folder_name)

# Take user input for the task to generate HTML code
instruction = input("💡 What should I build? (e.g., 'Create a web page with a welcome message'): ")

# Start the continuous interaction and code modification process
continuous_web_chat_and_update_code(instruction)

💡 What should I build? (e.g., 'Create a web page with a welcome message'): html code to make basic calculator


llama_perf_context_print:        load time =    2723.96 ms
llama_perf_context_print: prompt eval time =    2723.76 ms /    36 tokens (   75.66 ms per token,    13.22 tokens per second)
llama_perf_context_print:        eval time =  100658.16 ms /   511 runs   (  196.98 ms per token,     5.08 tokens per second)
llama_perf_context_print:       total time =  103947.99 ms /   547 tokens



🧠 Generated HTML Code:

Here is a basic HTML calculator:

```html
<!DOCTYPE html>
<html>
<head>
  <title>Basic Calculator</title>
</head>
<body>
  <form id="calculator">
    <input type="text" id="display" disabled>
    <br>
    <input type="button" value="7" onclick="append('7')">
    <input type="button" value="8" onclick="append('8')">
    <input type="button" value="9" onclick="append('9')">
    <input type="button" value="+">
    <br>
    <input type="button" value="4" onclick="append('4')">
    <input type="button" value="5" onclick="append('5')">
    <input type="button" value="6" onclick="append('6')">
    <input type="button" value="-">
    <br>
    <input type="button" value="1" onclick="append('1')">
    <input type="button" value="2" onclick="append('2')">
    <input type="button" value="3" onclick="append('3')">
    <input type="button" value="*">
    <br>
    <input type="button" value="0" onclick="append('0')">
    <input type="button" value="." onclick="append('.')">
 